<a href="https://colab.research.google.com/github/Tanish123-art/RAG/blob/main/recipes/RAG/Granite_Docling_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building an AI-Powered Document Retrieval System with Docling and Granite

*Using IBM Granite Models*

## Recipe Overview

Welcome to this Granite recipe, in this recipe, you'll learn to harness the power of advanced tools to build AI-powered document retrieval systems. It will guide you through:

- **Document Processing:** Learn to handle documents from various sources, parse and transform them into usable formats, and store them in vector databases using Docling.
- **Retrieval-Augmented Generation (RAG):** Understand how to connect large language models (LLMs) like Granite with external knowledge bases to enhance query responses and generate valuable insights.
- **LangChain for Workflow Integration:** Discover how to use LangChain to streamline and orchestrate document processing and retrieval workflows, enabling seamless interaction between different components of the system.

This recipe leverages three cutting-edge technologies:

1. **[Docling](https://docling-project.github.io/docling/):** An open-source toolkit for parsing and converting documents.
2. **[Granite](https://www.ibm.com/granite/docs/models/granite/):** A state-of-the-art LLM available via an [API](https://www.ibm.com/topics/api) through Replicate, providing robust natural language capabilities.
3. **[LangChain](https://github.com/langchain-ai/langchain):** A powerful framework for building applications powered by language models, designed to simplify complex workflows and integrate external tools seamlessly.

By the end of this recipe, you will:
- Gain proficiency in document processing and chunking.
- Integrate vector databases to enhance retrieval capabilities.
- Utilize RAG to perform efficient and accurate data retrieval for real-world applications.

This recipe is designed for AI developers, researchers, and enthusiasts looking to enhance their knowledge of document management and advanced NLP techniques.



## Prerequisites

- Familiarity with Python programming.
- Basic understanding of large language models and natural language processing concepts.


## Step 1: Setting up the environment

Install dependencies.

In [4]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils" \
    transformers \
    langchain_classic \
    langchain_core \
    langchain_huggingface sentence_transformers \
    langchain_chroma chromadb \
    docling \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git"
! echo "::endgroup::"

::group::Install Dependencies
Using Python 3.12.13 environment at: /usr
Resolved 166 packages in 1.14s
Checked 166 packages in 9ms
::endgroup::


## Step 2: Selecting System Components

### Choose your Embeddings Model

Specify the model to use for generating embedding vectors from text. Here we will be using one of the new [Granite Embeddings models](https://huggingface.co/collections/ibm-granite/granite-embedding-models-6750b30c802c1926a35550bb)

To use a model from another provider, replace this code cell with one from [this Embeddings Model recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Embeddings_Models.ipynb).

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import AutoTokenizer

embeddings_model_path = "ibm-granite/granite-embedding-small-english-r2"
embeddings_model = HuggingFaceEmbeddings(
    model_name=embeddings_model_path,
)
embeddings_tokenizer = AutoTokenizer.from_pretrained(embeddings_model_path)

Loading weights:   0%|          | 0/74 [00:00<?, ?it/s]

### Use the Granite model

Select a Granite model from the [`ibm-granite`](https://replicate.com/ibm-granite) org on Replicate. Here we use the Replicate Langchain client to connect to the model.

To get set up with Replicate, see [Getting Started with Replicate](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started/Getting_Started_with_Replicate.ipynb).

To connect to a model on a provider other than Replicate, substitute this code cell with one from the [LLM component recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_LLMs.ipynb).

In [6]:
from langchain_replicate import ChatReplicate
from ibm_granite_community.notebook_utils import get_env_var

model_path = "ibm-granite/granite-4.1-8b"
model = ChatReplicate(
    model=model_path,
    replicate_api_token=get_env_var("REPLICATE_API_TOKEN").strip(), # Added .strip() to remove potential whitespace
    model_kwargs={
        "max_tokens": 1000, # Set the maximum number of tokens to generate as output.
        "min_tokens": 100, # Set the minimum number of tokens to generate as output.
    },
)

REPLICATE_API_TOKEN loaded from Google Colab secret.


Now that we have the model downloaded, let's try asking it a question

In [7]:
from langchain_core.prompts import ChatPromptTemplate

query = "Who won in the Pantoja vs Asakura fight at UFC 310?"

# Create a Granite prompt for question-answering
prompt_template = ChatPromptTemplate.from_template(template="{input}")

chain = prompt_template | model

output = chain.invoke({"input": query})

print(output.text)

The fight between Jorge Pantoja and Yuki Asakura took place at UFC Fight Night: Pantoja vs. Asakura on September 23, 2023, not UFC 310. The bout was won by Jorge Pantoja via TKO (punches) in the second round. Pantoja demonstrated strong striking and grappling skills, ultimately securing the victory over Asakura. 

To clarify, UFC 310 does not exist in the official UFC event numbering, and the fight you're referring to occurred under the UFC Fight Night series.


Now, I know that UFC 310 happened in 2024, and this does not seem to be the right Pantoja. The model doesn't seem to know the answer but at least understands that this matchup did not occur. Let's see if it has some specific UFC rules info.

In [13]:
query1 = "How much weight allowance is allowed in non championship fights in the UFC?"

output = chain.invoke({"input": query1})

print(output.text)

In the Ultimate Fighting Championship (UFC), weight allowances for non-championship fights are generally the same as those for championship fights. The UFC follows the standard weight class system and the associated weight cut requirements set by the athletic commissions that oversee the events.

As of my last update, the UFC weight classes and their respective weight limits (including the catchweight for the flyweight division) are:

1. **Flyweight**: 125 lbs (56.7 kg)
2. **Bantamweight**: 135 lbs (61.2 kg)
3. **Featherweight**: 145 lbs (65.8 kg)
4. **Lightweight**: 155 lbs (70.3 kg)
5. **Welterweight**: 170 lbs (77.1 kg)
6. **Middleweight**: 185 lbs (83.9 kg)
7. **Light Heavyweight**: 205 lbs (93.0 kg)
8. **Heavyweight**: 265 lbs (120.2 kg) and above (up to 275 lbs for some events)

For non-championship fights, fighters must adhere to the same weight limits as they would for championship fights. However, there are a few nuances:

- **Catchweight**: In some cases, especially for non-t

Based on the official UFC rules, this is not 100% correct. Let's try getting some documents that contains this information for the model.

### Choose your Vector Database

Specify the database to use for storing and retrieving embedding vectors.

To connect to a vector database other than ChromaDB, replace this code cell with one from [this Vector Store recipe](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Components/Langchain_Vector_Stores.ipynb).

In [9]:
from langchain_chroma import Chroma

vector_db = Chroma(embedding_function=embeddings_model)

## Step 3: Building the Vector Database

In this example, from a set of source documents, we use [Docling](https://docling-project.github.io/docling/) to convert the documents into text and then split the text into chunks, derive embedding vectors using the embedding model, and load it into the vector database. Creating this vector database will allow us to easily search across our documents, enabling us to use RAG.

### Use Docling to download the documents, convert to text, and split into chunks

Here we have found a website that gives us information on UFC 310, as well as a PDF of the official UFC rules. Below, we will see that Docling can both convert and chunk the two documents.

In [10]:
# Docling imports
from docling.document_converter import DocumentConverter
from docling_core.transforms.chunker.hybrid_chunker import HybridChunker
from docling_core.types.doc.labels import DocItemLabel
from langchain_core.documents import Document

# Here are our documents, feel free to add more documents in formats that Docling supports
sources = [
    "https://www.ufc.com/news/main-card-results-highlights-winner-interviews-ufc-310-pantoja-vs-asakura",
    "https://www.abcboxing.com/wp-content/uploads/2020/02/unified-rules-mma-2019.pdf",
]

converter = DocumentConverter()

# Convert and chunk out documents
doc_id = 0
texts: list[Document] = [
    Document(page_content=chunk.text, metadata={"doc_id": (doc_id:=doc_id+1), "source": source})
    for source in sources
    for chunk in HybridChunker(tokenizer=embeddings_tokenizer).chunk(converter.convert(source=source).document)
    if any(filter(lambda c: c.label in [DocItemLabel.TEXT, DocItemLabel.PARAGRAPH], iter(chunk.meta.doc_items)))
]

print(f"{len(texts)} document chunks created")

[INFO] 2026-07-07 08:02:30,412 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-07 08:02:30,497 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-07 08:02:30,499 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.onnx
[INFO] 2026-07-07 08:02:30,642 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-07 08:02:30,649 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-07 08:02:30,650 [RapidOCR] main.py:63: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-07-07 08:02:30,765 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-07-07 08:02:30,913 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/l

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

32 document chunks created


In [11]:
# Print all created documents
for document in texts:
    print(f"Document ID: {document.metadata['doc_id']}")
    print(f"Source: {document.metadata['source']}")
    print(f"Content:\n{document.page_content}")
    print("=" * 80)  # Separator for clarity

Document ID: 1
Source: https://www.ufc.com/news/main-card-results-highlights-winner-interviews-ufc-310-pantoja-vs-asakura
Content:
- [Events](/events)
- [Upcoming](/events#events-list-upcoming)
- [Past](/events#events-list-past)
- [Tickets](/tickets)
- [VIP Experiences](https://ufcvip.com/?utm_source=ufc.com&utm_medium=referral&utm_campaign=vip_packages-main_menu_events_dropdown)
- [Group Sales](https://www.ufc.com/groupsales)
- [UFC Travel Deals](https://www.lucidtravel.com/team/events-public/ufc/16400)
- [Road to UFC](http://ufc.com/rtu)
- [Dana White's Contender Series](https://www.ufc.com/dwcs)
- [UFC BJJ](https://www.ufc.com/ufcbjj)
- [Rankings](/rankings)
- [Athletes](/athletes)
- [All Athletes](/athletes/all)
- [Hall of Fame](https://www.ufc.com/ufc-hall-of-fame)
- [Record Book](https://statleaders.ufc.com/)
- [News](/trending/all)
- [Connect](https://www.ufc.com/newsletter)
- [Newsletter](https://www.ufc.com/newsletter)
- [UFC Fight Club](https://ufcfightclub.com/)
- [UFC Apex]

### Populate the vector database

NOTE: Population of the vector database may take over a minute depending on your embedding model and service.

In [12]:
ids = vector_db.add_documents(texts[:5])
print(f"{len(ids)} documents added to the vector database")

5 documents added to the vector database


## Step 4: RAG with Granite

Now that we have succesfully converted our documents and vectorized them, we can set up out RAG pipeline.

### Retrieve relevant chunks



Here we will test the as_retriever method to search through our newly created vector database for chunks that are relevant to our original query



In [14]:
retriever = vector_db.as_retriever()

docs = retriever.invoke(query)
print(docs)

[Document(id='85838f15-487a-466c-95ac-7233aa3ce002', metadata={'source': 'https://www.ufc.com/news/main-card-results-highlights-winner-interviews-ufc-310-pantoja-vs-asakura', 'doc_id': 2}, page_content='See The Fight Results, Watch Post-Fight Interviews With The Main Card Winners And More From UFC 310: Pantoja vs Asakura, Live From T-Mobile Arena In Las Vegas, Nevada\nBy E. Spencer Kyte, On X @spencerkyte • Dec. 8, 2024\nThe UFC 310 preliminary card slate was outstanding, featuring six finishes and trio of entertaining three-round battles, setting the stage for a captivating pay-per-view main card at T-Mobile Arena in Las Vegas.\nAnd the action in the Octagon delivered in a massive way.\nDooho Choi kicked off the festivities with a standout performance against Nate Landwehr, finishing from a mounted crucifix in the third round before Bryce Mitchell followed suit one fight later, putting Kron Gracie to sleep with a pair of thudding elbows from inside his guard. After heavyweight contend

Looks like it pulled some chunks that would have the information we are looking for. Let's go ahead and contruct our RAG pipeline.

### Create the prompt for Granite

Next, we construct the prompt pipeline. This creates the prompt which holds the retrieved chunks from out previous search and feeds this to the model as context for answering our question.

In [15]:
from ibm_granite_community.langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

# Assemble the retrieval-augmented generation chain
combine_docs_chain = create_stuff_documents_chain(
    llm=model,
    prompt=prompt_template,
)
rag_chain = create_retrieval_chain(
    retriever=vector_db.as_retriever(),
    combine_docs_chain=combine_docs_chain,
)

### Generate a retrieval-augmented response to a question

The pipeline uses the query to locate documents from the vector database and use them as context for the query.

In [16]:
output = rag_chain.invoke({"input": query})

print(output['answer'])

Alexandre Pantoja won the fight against Kai Asakura by submission (rear-naked choke) at 2:05 of Round 2. This information is clearly stated in the document under the "Main Event" section. 

**Final answer:** Alexandre Pantoja. 

**Confidence:** 100% (The answer is directly provided in the text.) 

**Reasoning:** The document explicitly lists the outcome of the main event, specifying that Alexandre Pantoja defeated Kai Asakura by rear-naked choke in the second round. No other interpretations or additional context are needed to answer the question.


Awesome! It looks like the model figured out our first question. Let's see if it figure out the rule we were looking for.

In [17]:
output = rag_chain.invoke({"input": query1})

print(output['answer'])

The provided context does not contain any information regarding weight allowances in non-championship UFC fights. The documents focus on fight results, interviews, and event details for UFC 310, but they do not mention specific weight allowance rules for non-championship bouts. Therefore, the question cannot be answered based on the available information. 

To answer this question, one would need to refer to the official UFC weight class rules or regulations, which specify the weight limits and allowances for both championship and non-championship fights. These rules are typically outlined in the UFC's official fighter contracts or promotional materials.


Awesome! We can now see that we have created a pipeline that can successfully leverage knowledge from multiple document types for generation.

## Next Steps

- Explore advanced RAG workflows for other industries
- Experiment with other document types and larger datasets.
- Optimize prompt engineering for better Granite responses.

Thank you for using this recipe!